In [0]:
dbutils.widgets.dropdown("use_unity_catalog", "true", ["true", "false"], "Use Unity Catalog")
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_prefix", "retail", "Schema Prefix")

In [0]:
catalog = dbutils.widgets.get("catalog_name")
schema_prefix = dbutils.widgets.get("schema_prefix")
bronze_db = f"{catalog}.{schema_prefix}_bronze"
silver_db = f"{catalog}.{schema_prefix}_silver"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {silver_db}")
print("Bronze:", bronze_db, "| Silver:", silver_db)

In [0]:
%run ./00b_pipeline_utils

In [0]:
from pyspark.sql.functions import col, trim, upper, lower, coalesce, lit, mean, first, initcap
from delta.tables import DeltaTable

def merge_into_silver(df, table_name, join_keys):
    target = f"{silver_db}.{table_name}"
    deduped = df.dropDuplicates(join_keys)

    if not spark.catalog.tableExists(target):
        deduped.write.format("delta").mode("overwrite").saveAsTable(target)
        print(f"{target}: created with {deduped.count()} rows")
    else:
        target_delta = DeltaTable.forName(spark, target)
        condition = " AND ".join([f"target.{k} = source.{k}" for k in join_keys])
        (target_delta.alias("target")
         .merge(deduped.alias("source"), condition)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
        print(f"{target}: merged, now {spark.table(target).count()} rows total")

In [0]:
customers_raw = spark.table(f"{bronze_db}.raw_customers")
customers_typed = (customers_raw
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_unique_id", trim(col("customer_unique_id")))
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("string"))
    .withColumn("customer_city", initcap(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state")))))
customers_clean = quarantine_records(customers_typed, col("customer_id").isNotNull(), "silver_customers", "missing customer_id")
merge_into_silver(customers_clean, "customers", ["customer_id"])

sellers_raw = spark.table(f"{bronze_db}.raw_sellers")
sellers_typed = (sellers_raw
    .withColumn("seller_id", trim(col("seller_id")))
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("string"))
    .withColumn("seller_city", initcap(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state")))))
sellers_clean = quarantine_records(sellers_typed, col("seller_id").isNotNull(), "silver_sellers", "missing seller_id")
merge_into_silver(sellers_clean, "sellers", ["seller_id"])

products_raw = spark.table(f"{bronze_db}.raw_products")
products_typed = (products_raw
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_category_name", coalesce(trim(col("product_category_name")), lit("unknown")))
    .withColumn("product_weight_g", coalesce(col("product_weight_g").cast("double"), lit(0.0)))
    .withColumn("product_length_cm", coalesce(col("product_length_cm").cast("double"), lit(0.0)))
    .withColumn("product_height_cm", coalesce(col("product_height_cm").cast("double"), lit(0.0)))
    .withColumn("product_width_cm", coalesce(col("product_width_cm").cast("double"), lit(0.0))))
products_clean = quarantine_records(products_typed, col("product_id").isNotNull(), "silver_products", "missing product_id")
merge_into_silver(products_clean, "products", ["product_id"])

In [0]:
category_raw = spark.table(f"{bronze_db}.raw_product_category_name_translation")
category_typed = (category_raw
    .withColumn("product_category_name", trim(col("product_category_name")))
    .withColumn("product_category_name_english", trim(col("product_category_name_english"))))
category_clean = quarantine_records(category_typed, col("product_category_name").isNotNull(), "silver_category_translation", "missing product_category_name")
merge_into_silver(category_clean, "product_category_name_translation", ["product_category_name"])

orders_raw = spark.table(f"{bronze_db}.raw_orders")
orders_typed = (orders_raw
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp"))
    .withColumn("order_approved_at", col("order_approved_at").cast("timestamp"))
    .withColumn("order_delivered_carrier_date", col("order_delivered_carrier_date").cast("timestamp"))
    .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp"))
    .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("timestamp")))
orders_clean = quarantine_records(orders_typed, col("order_id").isNotNull() & col("customer_id").isNotNull(), "silver_orders", "missing order_id or customer_id")
merge_into_silver(orders_clean, "orders", ["order_id"])

items_raw = spark.table(f"{bronze_db}.raw_order_items")
items_typed = (items_raw
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("order_item_id", col("order_item_id").cast("integer"))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("seller_id", trim(col("seller_id")))
    .withColumn("shipping_limit_date", col("shipping_limit_date").cast("timestamp"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double")))

items_stage1 = quarantine_records(items_typed, col("order_id").isNotNull() & col("order_item_id").isNotNull(), "silver_order_items", "missing order_id/order_item_id")
items_stage2 = quarantine_records(items_stage1, col("price") >= 0.0, "silver_order_items", "negative price")
items_clean = quarantine_records(items_stage2, col("freight_value") >= 0.0, "silver_order_items", "negative freight_value")
merge_into_silver(items_clean, "order_items", ["order_id", "order_item_id"])

In [0]:
payments_raw = spark.table(f"{bronze_db}.raw_order_payments")
payments_typed = (payments_raw
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("payment_sequential", col("payment_sequential").cast("integer"))
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("payment_installments", col("payment_installments").cast("integer"))
    .withColumn("payment_value", col("payment_value").cast("double")))
payments_stage1 = quarantine_records(payments_typed, col("order_id").isNotNull() & col("payment_sequential").isNotNull(), "silver_order_payments", "missing order_id/payment_sequential")
payments_clean = quarantine_records(payments_stage1, col("payment_value") >= 0.0, "silver_order_payments", "negative payment_value")
merge_into_silver(payments_clean, "order_payments", ["order_id", "payment_sequential"])

reviews_raw = spark.table(f"{bronze_db}.raw_order_reviews")
reviews_typed = (reviews_raw
    .withColumn("review_id", trim(col("review_id")))
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("review_score", col("review_score").cast("integer"))
    .withColumn("review_comment_title", trim(col("review_comment_title")))
    .withColumn("review_comment_message", trim(col("review_comment_message")))
    .withColumn("review_creation_date", col("review_creation_date").cast("timestamp"))
    .withColumn("review_answer_timestamp", col("review_answer_timestamp").cast("timestamp")))
reviews_clean = quarantine_records(reviews_typed, col("review_id").isNotNull() & col("order_id").isNotNull(), "silver_order_reviews", "missing review_id or order_id")
merge_into_silver(reviews_clean, "order_reviews", ["review_id", "order_id"])

geo_raw = spark.table(f"{bronze_db}.raw_geolocation")
geo_valid = quarantine_records(geo_raw, col("geolocation_zip_code_prefix").isNotNull(), "silver_geolocation", "missing geolocation_zip_code_prefix")
geolocation_agg = (geo_valid
    .withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("string"))
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))
    .withColumn("geolocation_city", initcap(trim(col("geolocation_city"))))
    .withColumn("geolocation_state", upper(trim(col("geolocation_state"))))
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        mean("geolocation_lat").alias("geolocation_lat"),
        mean("geolocation_lng").alias("geolocation_lng"),
        first("geolocation_city").alias("geolocation_city"),
        first("geolocation_state").alias("geolocation_state")
    ))
merge_into_silver(geolocation_agg, "geolocation", ["geolocation_zip_code_prefix"])

In [0]:
display(spark.sql(f"SHOW TABLES IN {silver_db}"))
display(spark.sql(f"SELECT * FROM {catalog}.retail_ops.pipeline_logs WHERE layer = 'quarantine' ORDER BY log_timestamp DESC"))